In [ ]:
from pathlib import Path
from itertools import combinations
import warnings

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    recall_score,
    precision_score,
    brier_score_loss
)


In [ ]:
# ============================================================
# RUNTIME BENCHMARK LOGGER
# ============================================================

from time import perf_counter
from datetime import datetime
from pathlib import Path
import pandas as pd
import os

RUNTIME_FILE = Path("FINAL_experimental_runtime_summary.csv")

if RUNTIME_FILE.exists():
    RUNTIME_LOG = pd.read_csv(RUNTIME_FILE).to_dict("records")
else:
    RUNTIME_LOG = []


def start_timer(experiment_name):
    print("\n" + "=" * 75)
    print(f"STARTING: {experiment_name}")
    print("Clock time:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    print("=" * 75)

    return perf_counter()


def stop_timer(experiment_name, start_time):
    elapsed_sec = perf_counter() - start_time

    record = {
        "experiment": experiment_name,
        "runtime_seconds": elapsed_sec,
        "runtime_minutes": elapsed_sec / 60.0,
        "runtime_hours": elapsed_sec / 3600.0,
    }

    RUNTIME_LOG.append(record)

    pd.DataFrame(RUNTIME_LOG).to_csv(
        RUNTIME_FILE,
        index=False
    )

    print("\n" + "-" * 75)
    print(f"FINISHED: {experiment_name}")
    print(f"Runtime: {elapsed_sec:.2f} seconds")
    print(f"Runtime: {elapsed_sec / 60.0:.4f} minutes")
    print("-" * 75)

    return elapsed_sec

In [ ]:
# Robust project paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "Data" / "Raw"
OUT_DIR = PROJECT_ROOT / "Results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = RAW_DIR / "triage_cleaned_working.csv"
DIAG_PATH = RAW_DIR / "triage_train_diagnostics.csv"
REGISTRY_PATH = RAW_DIR / "approved_slice_registry_FINAL.csv"

for p in [DATA_PATH, DIAG_PATH, REGISTRY_PATH]:
    print(p.resolve(), "exists:", p.exists())

df = pd.read_csv(DATA_PATH)
diag_train = pd.read_csv(DIAG_PATH)
basic_registry = pd.read_csv(REGISTRY_PATH)

print("df:", df.shape)
print("diag_train:", diag_train.shape)
print("basic_registry:", basic_registry.shape)


In [ ]:
#df = pd.read_csv("../Data/Raw/triage_cleaned_working.csv")
#diag_train = pd.read_csv("../Data/Raw/triage_train_diagnostics.csv")
#basic_registry = pd.read_csv("../Data/Raw/approved_slice_registry_FINAL.csv")

In [ ]:
MIN_SUPPORT_N = int(0.005 * len(diag_train))  # 0.5%
print("MIN_SUPPORT_N:", MIN_SUPPORT_N)

common helper functions

In [ ]:
def evaluate_slice(data, mask, method, slice_name, threshold=0.5, min_n=MIN_SUPPORT_N, min_pos=1):
    """
    Evaluate one slice. Returns None for tiny slices or slices with no positive labels.
    This avoids sklearn warnings such as:
    'No positive class found in y_true'.
    """
    mask = pd.Series(mask, index=data.index).astype(bool)
    sub = data.loc[mask].copy()

    if len(sub) < min_n:
        return None

    required = ["y", "oof_pred_proba", "oof_logloss", "false_negative_05", "high_conf_contradiction"]
    missing = [c for c in required if c not in sub.columns]
    if missing:
        raise KeyError(f"Missing required diagnostic columns in data: {missing}")

    y_true = sub["y"].astype(int).values
    if y_true.sum() < min_pos:
        return None

    p = sub["oof_pred_proba"].astype(float).clip(1e-6, 1 - 1e-6).values
    pred = (p >= threshold).astype(int)

    return {
        "method": method,
        "slice_name": slice_name,
        "n": int(len(sub)),
        "support_pct": 100 * len(sub) / len(data),
        "event_rate": float(y_true.mean()),
        "AUROC": roc_auc_score(y_true, p) if len(np.unique(y_true)) > 1 else np.nan,
        "AUPRC": average_precision_score(y_true, p),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Brier": brier_score_loss(y_true, p),
        "avg_logloss": sub["oof_logloss"].mean(),
        "fn_rate": sub["false_negative_05"].mean(),
        "hcc_rate": sub["high_conf_contradiction"].mean()
    }

Define global reference metrics

In [ ]:
# Global reference metrics
global_ref = evaluate_slice(
    diag_train,
    pd.Series(True, index=diag_train.index),
    method="GLOBAL",
    slice_name="GLOBAL",
    min_n=MIN_SUPPORT_N,
    min_pos=1
)

global_ref


In [ ]:
print("GLOBAL METRICS USED IN 05b")
print(global_ref)

print("\nTemperature summary:")
print(diag_train["temperature"].describe())
print("fever_temp_flag count:", (diag_train["temperature"] > 38.5).sum())

In [ ]:
#MIN_SUPPORT_N = int(0.005 * len(diag_train))  # 0.5%
#print("MIN_SUPPORT_N:", MIN_SUPPORT_N)

In [ ]:
#ef add_harm_columns(results_df, global_ref):
#   results_df["AUPRC_gap"] = global_ref["AUPRC"] - results_df["AUPRC"]
#   results_df["Recall_gap"] = global_ref["Recall"] - results_df["Recall"]
#   results_df["LogLoss_lift"] = results_df["avg_logloss"] - global_ref["avg_logloss"]
#   results_df["harm_score"] = (
#       results_df["LogLoss_lift"].fillna(0)
#       + results_df["AUPRC_gap"].fillna(0)
#       + results_df["Recall_gap"].fillna(0)
#   )
#   return results_df

In [ ]:
#ef add_harm_columns(results_df, global_ref):
#   results_df = results_df.copy()
#
#   results_df["AUPRC_gap"] = global_ref["AUPRC"] - results_df["AUPRC"]
#   results_df["Recall_gap"] = global_ref["Recall"] - results_df["Recall"]
#   results_df["LogLoss_lift"] = results_df["avg_logloss"] - global_ref["avg_logloss"]
#   results_df["FN_lift"] = results_df["fn_rate"] - global_ref["fn_rate"]
#   results_df["AUPRC_lift_over_prevalence"] = results_df["AUPRC"] - results_df["event_rate"]
#
#   # Repair-oriented harm: prioritize genuinely difficult, missed-admission, high-loss slices
#   results_df["harm_score"] = (
#       results_df["LogLoss_lift"].clip(lower=0)
#       + results_df["FN_lift"].clip(lower=0)
#       - results_df["AUPRC_lift_over_prevalence"].clip(upper=0)
#   )
#
#   return results_df

In [ ]:
# ============================================================
# COMPARATIVE COMPOSITE HARM
# Must match Notebook 8 and Section III
#
# H_k = ΔL_k + ΔFN_k + ΔHCC_k
# ============================================================

import numpy as np
import pandas as pd


def add_harm_columns(
    results_df,
    global_ref,
):
    results_df = (
        results_df.copy()
    )

    required_columns = [
        "avg_logloss",
        "fn_rate",
        "hcc_rate",
        "AUPRC",
        "Recall",
        "event_rate",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in results_df.columns
    ]

    if missing_columns:
        raise KeyError(
            "Comparative result table is missing: "
            f"{missing_columns}"
        )

    required_global_metrics = [
        "avg_logloss",
        "fn_rate",
        "hcc_rate",
        "AUPRC",
        "Recall",
    ]

    missing_global_metrics = [
        metric
        for metric in required_global_metrics
        if metric not in global_ref
    ]

    if missing_global_metrics:
        raise KeyError(
            "global_ref is missing: "
            f"{missing_global_metrics}"
        )

    results_df["AUPRC_gap"] = (
        global_ref["AUPRC"]
        -
        results_df["AUPRC"]
    )

    results_df["Recall_gap"] = (
        global_ref["Recall"]
        -
        results_df["Recall"]
    )

    results_df["LogLoss_lift"] = (
        results_df["avg_logloss"]
        -
        global_ref["avg_logloss"]
    )

    results_df["FN_lift"] = (
        results_df["fn_rate"]
        -
        global_ref["fn_rate"]
    )

    results_df["HCC_lift"] = (
        results_df["hcc_rate"]
        -
        global_ref["hcc_rate"]
    )

    results_df[
        "AUPRC_lift_over_prevalence"
    ] = (
        results_df["AUPRC"]
        -
        results_df["event_rate"]
    )

    # The same unweighted composite used in Notebook 8.
    results_df["harm_score"] = (
        results_df["LogLoss_lift"]
        +
        results_df["FN_lift"]
        +
        results_df["HCC_lift"]
    )

    return results_df

BASiC/ASAP

In [ ]:
basic_slices = basic_registry[
    basic_registry["final_accept"].astype(str).str.lower().isin(["yes", "true", "1"])
]["slice_name"].tolist()

Add clinical plausibility scoring

In [ ]:
def plausibility_score(slice_name):
    name = str(slice_name).lower()

    if any(x in name for x in [
        "tachy", "hypoxia", "pneumonia", "ckd",
        "chest", "abdominal", "spo2", "hr_missing",
        "tachypnea", "hypotension"
    ]):
        return 1.0

    if any(x in name for x in [
        "arrival", "vitals_missing", "unclassified",
        "missing", "temperature"
    ]):
        return 0.7

    return 0.4

In [ ]:
#def plausibility_score(slice_name):
#    name = slice_name.lower()
#    if any(x in name for x in ["tachy", "hypoxia", "pneumonia", "ckd", "chest", "abdominal", "spo2", "hr_missing"]):
#        return 1.0
#    if any(x in name for x in ["arrival", "vitals_missing", "unclassified"]):
#        return 0.7
#    return 0.4
#
#all_results["clinical_plausibility"] = all_results["slice_name"].apply(plausibility_score)

slice_membership function from nb_08

In [ ]:
def add_derived_slice_columns(df):
    """
    Add all derived columns needed by BASiC/ASAP and SliceFinder-style rules.
    Run this after every reload/kernel restart.
    """
    df = df.copy()

    arrival = df["arrival_mode"].astype(str).str.lower().str.strip()

    df["arrival_mode_ambulance"] = arrival.eq("ambulance").astype(int)
    df["arrival_mode_wheelchair"] = arrival.eq("wheelchair").astype(int)
    df["arrival_mode_unknown"] = arrival.eq("unknown").astype(int)

    df["tachycardia_flag"] = (
    df["heart_rate_final"].notna() & (df["heart_rate_final"] > 120)
    ).astype(int)
    
    df["hypotension_flag"] = (
        df["bp_systolic"].notna() & (df["bp_systolic"] < 90)
    ).astype(int)
    
    df["tachypnea_flag"] = (
        df["respiratory"].notna() & (df["respiratory"] > 24)
    ).astype(int)
    
    df["hypoxia_flag"] = (
        df["spo2"].notna() & (df["spo2"] < 92)
    ).astype(int)
    
    temp_c = (df["temperature"] - 32) * 5 / 9

    df["fever_temp_flag"] = (
        temp_c.notna() &
        (temp_c > 38.5)
    ).astype(int)
    
    df["hypothermia_flag"] = (
        temp_c.notna() &
        (temp_c < 35)
    ).astype(int)

    return df


def build_slice_membership(df, registry):
    """
    Convert accepted slice names in the registry into a binary membership matrix.
    The registry says WHICH slices to use; slice_rules says HOW to compute them.
    """
    df = add_derived_slice_columns(df)

    slice_rules = {
        # Single complaint slices
        "chest_pain": lambda d: d["chest_pain"] == 1,
        "abdominal_pain": lambda d: d["abdominal_pain"] == 1,
        "back_pain": lambda d: d["back_pain"] == 1,
        "headache": lambda d: d["headache"] == 1,
        "cough": lambda d: d["cough"] == 1,
        "unclassified": lambda d: d["unclassified"] == 1,

        # Comorbidity slices
        "ckd": lambda d: d["ckd"] == 1,
        "dm": lambda d: d["dm"] == 1,
        "htn": lambda d: d["htn"] == 1,
        "chd": lambda d: d["chd"] == 1,
        "pneumonia": lambda d: d["pneumonia"] == 1,

        # Operational slices
        "arrival_mode_ambulance": lambda d: d["arrival_mode_ambulance"] == 1,
        "arrival_mode_wheelchair": lambda d: d["arrival_mode_wheelchair"] == 1,
        "arrival_mode_unknown": lambda d: d["arrival_mode_unknown"] == 1,
        "vitals_missing_ge_3": lambda d: d["vitals_missing_count"] >= 3,
        "spo2_missing": lambda d: d["spo2_missing"] == 1,
        "hr_missing": lambda d: d["hr_missing"] == 1,

        # Physiology slices
        "tachycardia": lambda d: d["tachycardia_flag"] == 1,
        "hypotension": lambda d: d["hypotension_flag"] == 1,
        "tachypnea": lambda d: d["tachypnea_flag"] == 1,
        "hypoxia": lambda d: d["hypoxia_flag"] == 1,
        "fever_temp": lambda d: d["fever_temp_flag"] == 1,
        "hypothermia": lambda d: d["hypothermia_flag"] == 1,

        # Degree-2 clinical slices
        "chest_pain_and_ambulance": lambda d: (
            (d["chest_pain"] == 1) & (d["arrival_mode_ambulance"] == 1)
        ),
        "abdominal_pain_and_hr_missing": lambda d: (
            (d["abdominal_pain"] == 1) & (d["hr_missing"] == 1)
        ),
        "ckd_and_spo2_missing": lambda d: (
            (d["ckd"] == 1) & (d["spo2_missing"] == 1)
        ),
        "pneumonia_and_tachypnea": lambda d: (
            (d["pneumonia"] == 1) & (d["tachypnea_flag"] == 1)
        ),
        "unclassified_and_vitals_missing_ge_3": lambda d: (
            (d["unclassified"] == 1) & (d["vitals_missing_count"] >= 3)
        ),

        # Error-guided aliases
        "err_vitals_missing_ge_3": lambda d: d["vitals_missing_count"] >= 3,
        "err_spo2_missing_and_unclassified": lambda d: (
            (d["spo2_missing"] == 1) & (d["unclassified"] == 1)
        ),
        "err_hr_missing_and_abdominal_pain": lambda d: (
            (d["hr_missing"] == 1) & (d["abdominal_pain"] == 1)
        ),
        "err_pneumonia_and_tachypnea": lambda d: (
            (d["pneumonia"] == 1) & (d["tachypnea_flag"] == 1)
        ),
        "err_ckd_and_spo2_missing": lambda d: (
            (d["ckd"] == 1) & (d["spo2_missing"] == 1)
        ),
        "err_chest_pain_and_tachycardia": lambda d: (
            (d["chest_pain"] == 1) & (d["tachycardia_flag"] == 1)
        ),
        "err_unclassified_and_missingness": lambda d: (
            (d["unclassified"] == 1) & (d["vitals_missing_count"] >= 2)
        ),
        "err_hypoxia_and_arrival_missing": lambda d: (
            (d["hypoxia_flag"] == 1) & (d["arrival_mode_missing"] == 1)
        ),
    }

    membership = {}
    skipped = []

    for _, row in registry.iterrows():
        name = row["slice_name"]

        if name in slice_rules:
            membership[name] = slice_rules[name](df).astype(int).values
        elif name in df.columns:
            membership[name] = df[name].astype(int).values
        else:
            skipped.append(name)

    if skipped:
        print("Skipped unknown slice names:")
        for s in skipped:
            print(" -", s)

    return pd.DataFrame(membership, index=df.index)


In [ ]:
print("fever count:", df["fever_temp_flag"].sum())
print("hypothermia count:", df["hypothermia_flag"].sum())

In [ ]:
print(DIAG_PATH)
print(diag_train.shape)
print(diag_train.columns.tolist())

Build BASiC/ASAP membership matrix and BASiC result table

In [ ]:
# Keep only accepted slices from registry
accepted_registry = basic_registry[
    basic_registry["final_accept"].astype(str).str.lower().isin(["yes", "true", "1"])
].copy()

basic_slices = accepted_registry["slice_name"].dropna().tolist()

# Build membership matrix for accepted BASiC/ASAP slices
slice_membership = build_slice_membership(diag_train, accepted_registry)

print("Accepted registry slices:", len(basic_slices))
print("Membership matrix:", slice_membership.shape)
print("Membership columns:", list(slice_membership.columns))

# Evaluate BASiC/ASAP accepted slices
basic_rows = []

for s in basic_slices:
    if s not in slice_membership.columns:
        print("Skipping missing membership:", s)
        continue

    mask = slice_membership[s] == 1
    m = evaluate_slice(diag_train, mask, "BASiC_ASAP", s)

    if m is not None:
        basic_rows.append(m)

basic_results = pd.DataFrame(basic_rows)
basic_results = add_harm_columns(basic_results, global_ref)

basic_results.to_csv(OUT_DIR / "comparative_basic_asap_results.csv", index=False)

basic_results.sort_values("harm_score", ascending=False).head(10)


In [ ]:
# ============================================================
# BASiC CANDIDATE POOL FOR FAIR COMPLETE-ASAP COMPARISON
#
# IMPORTANT:
# Do not prefilter BASiC using the old repairability rule.
# The common complete-ASAP cell will evaluate governance
# eligibility consistently across all methods.
# ============================================================

basic_results = (
    basic_results.copy()
)

basic_results["clinical_plausibility"] = (
    basic_results["slice_name"]
    .apply(
        plausibility_score
    )
)

# Keep the complete BASiC candidate output.
# Complete ASAP is applied later, jointly with all competitors.
basic_comparison_results = (
    basic_results.copy()
)

basic_comparison_results["method"] = (
    "BASiC_ASAP"
)

print(
    "BASiC candidates entering common comparison:",
    len(basic_comparison_results),
)

display(
    basic_comparison_results[
        [
            "method",
            "slice_name",
            "n",
            "harm_score",
            "clinical_plausibility",
        ]
    ]
    .sort_values(
        "harm_score",
        ascending=False,
    )
)

Slice Finder approximation

In [ ]:
t_runtime_sf = start_timer(
    "SliceFinder-style discovery"
)
# Add derived columns directly into diag_train for SliceFinder-style candidate search
diag_train = add_derived_slice_columns(diag_train)

derived_check_cols = [
    "arrival_mode_ambulance", "arrival_mode_wheelchair", "arrival_mode_unknown",
    "tachycardia_flag", "hypotension_flag", "tachypnea_flag",
    "hypoxia_flag", "fever_temp_flag", "hypothermia_flag"
]

print(diag_train[derived_check_cols].sum())


In [ ]:
candidate_cols = [
    "abdominal_pain", "chest_pain", "back_pain", "headache", "cough",
    "ckd", "dm", "htn", "chd", "pneumonia",
    "hr_missing", "spo2_missing", "temperature_missing",
    "tachycardia_flag", "tachypnea_flag", "hypoxia_flag",
    "arrival_mode_ambulance", "arrival_mode_unknown"
]

Create single and pairwise slices

In [ ]:
# Already handled by add_derived_slice_columns above.
# Keep this cell as a safety rerun point.
diag_train = add_derived_slice_columns(diag_train)

print(diag_train[
    ["tachycardia_flag", "hypotension_flag", "tachypnea_flag",
     "hypoxia_flag", "fever_temp_flag", "hypothermia_flag"]
].sum())


In [ ]:
valid_candidate_cols = [c for c in candidate_cols if c in diag_train.columns]
missing_candidate_cols = [c for c in candidate_cols if c not in diag_train.columns]

print("Missing candidate columns:", missing_candidate_cols)
print("Valid candidate columns:", len(valid_candidate_cols))

slice_finder_rows = []

for c in valid_candidate_cols:
    mask = diag_train[c] == 1
    m = evaluate_slice(diag_train, mask, "SliceFinder_style", c)
    if m:
        slice_finder_rows.append(m)

from itertools import combinations

for c1, c2 in combinations(valid_candidate_cols, 2):
    mask = (diag_train[c1] == 1) & (diag_train[c2] == 1)
    name = f"{c1}_AND_{c2}"
    m = evaluate_slice(diag_train, mask, "SliceFinder_style", name)
    if m:
        slice_finder_rows.append(m)
slice_finder_results = pd.DataFrame(slice_finder_rows)

if len(slice_finder_results) > 0:
    slice_finder_results = add_harm_columns(slice_finder_results, global_ref)

slice_finder_results.to_csv(
    OUT_DIR / "comparative_slicefinder_style_results.csv",
    index=False
)

print("slice_finder_results:", slice_finder_results.shape)
slice_finder_results.sort_values("harm_score", ascending=False).head(10)
stop_timer(
    "SliceFinder-style discovery",
    t_runtime_sf
)

SliceLine approximation

In [ ]:
t_runtime_sl = start_timer(
    "SliceLine-style discovery"
)
MIN_SUPPORT = 0.005 * len(diag_train)

sliceline_rows = []

# first pass: keep only useful single slices
surviving_cols = []

for c in candidate_cols:
    mask = diag_train[c] == 1
    if mask.sum() < MIN_SUPPORT:
        continue

    m = evaluate_slice(diag_train, mask, "SliceLine_style", c)
    if m is None:
        continue

    if m["avg_logloss"] > global_ref["avg_logloss"]:
        surviving_cols.append(c)
        sliceline_rows.append(m)

# second pass: only combine surviving columns
for c1, c2 in combinations(surviving_cols, 2):
    mask = (diag_train[c1] == 1) & (diag_train[c2] == 1)
    if mask.sum() < MIN_SUPPORT:
        continue

    name = f"{c1}_AND_{c2}"
    m = evaluate_slice(diag_train, mask, "SliceLine_style", name)
    if m:
        sliceline_rows.append(m)

In [ ]:
sliceline_results = pd.DataFrame(sliceline_rows)

if len(sliceline_results) > 0:
    sliceline_results = add_harm_columns(sliceline_results, global_ref)

sliceline_results.to_csv(
    OUT_DIR / "comparative_sliceline_style_results.csv",
    index=False
)

print("sliceline_results:", sliceline_results.shape)
sliceline_results.sort_values("harm_score", ascending=False).head(10)
stop_timer(
    "SliceLine-style discovery",
    t_runtime_sl
)

AFISP-inspired implementation

In [ ]:
t_runtime_afisp = start_timer(
    "AFISP-inspired discovery"
)
afisp_defs = {
    "physiologic_instability": lambda d: (
        (d["tachycardia_flag"] == 1) |
        (d["tachypnea_flag"] == 1) |
        (d["hypoxia_flag"] == 1)
    ),
    "missing_vitals": lambda d: d["vitals_missing_count"] >= 3,
    "cardiorespiratory_risk": lambda d: (
        (d["chest_pain"] == 1) |
        (d["pneumonia"] == 1) |
        (d["chd"] == 1)
    ),
    "metabolic_comorbidity": lambda d: (
        (d["dm"] == 1) |
        (d["ckd"] == 1)
    ),
    "complaint_missingness": lambda d: (
        ((d["abdominal_pain"] == 1) | (d["chest_pain"] == 1)) &
        (d["vitals_missing_count"] >= 1)
    )
}

In [ ]:
afisp_rows = []

for name, rule in afisp_defs.items():
    mask = rule(diag_train)
    m = evaluate_slice(diag_train, mask, "AFISP_inspired", name)
    if m:
        afisp_rows.append(m)

In [ ]:
afisp_results = pd.DataFrame(afisp_rows)

if len(afisp_results) > 0:
    afisp_results = add_harm_columns(afisp_results, global_ref)

afisp_results.to_csv(
    OUT_DIR / "comparative_afisp_inspired_results.csv",
    index=False
)

print("afisp_results:", afisp_results.shape)
afisp_results.sort_values("harm_score", ascending=False).head(10)
stop_timer(
    "AFISP-inspired discovery",
    t_runtime_afisp
)

Combine all method results

In [ ]:
result_frames = []

# Keep BASiC repair-oriented version as the main BASiC result
for name in [
    "basic_comparison_results",
    "slice_finder_results",
    "sliceline_results",
    "afisp_results"
]:
    if name in globals():
        obj = globals()[name]
        if isinstance(obj, pd.DataFrame) and len(obj) > 0:
            result_frames.append(obj)
        else:
            print(f"Skipping {name}: empty DataFrame")
    else:
        print(f"Skipping {name}: not defined")

all_results = pd.concat(result_frames, ignore_index=True)

all_results = add_harm_columns(all_results, global_ref)
all_results["clinical_plausibility"] = all_results["slice_name"].apply(plausibility_score)

all_results.to_csv(
    OUT_DIR / "comparative_all_slice_discovery_results.csv",
    index=False
)

print(all_results.shape)
all_results.head()

Create Table X for Section VI

In [ ]:
#summary_rows = []
#
#for method, g in all_results.groupby("method"):
#    harmful = g[g["harm_score"] > 0]
#    repairable_harmful = g[
#        (g["harm_score"] > 0) &
#        (g["clinical_plausibility"] >= 0.7)
#    ]
#
#    summary_rows.append({
#        "method": method,
#        "n_slices": len(g),
#        "n_harmful_slices": len(harmful),
#        "repairable_harmful_slices": len(repairable_harmful),
#        "worst_slice_AUPRC": g["AUPRC"].min(),
#        "worst_slice_recall": g["Recall"].min(),
#        "mean_harm_score": g["harm_score"].mean(),
#        "max_harm_score": g["harm_score"].max(),
#        "mean_clinical_plausibility": g["clinical_plausibility"].mean(),
#        "coverage_pct_naive": g["support_pct"].sum(),
#        "top_slice": g.sort_values("harm_score", ascending=False).iloc[0]["slice_name"]
#    })
#
#comparative_summary = pd.DataFrame(summary_rows)
#
#comparative_summary.to_csv(
#    OUT_DIR / "table_X_comparative_method_summary.csv",
#    index=False
#)
#
#comparative_summary

In [ ]:
#TOP_K = 10
#
#summary_rows = []
#
#for method, g in all_results.groupby("method"):
#    g_top = g.sort_values("harm_score", ascending=False).head(TOP_K)
#
#    repairable_harmful = g_top[
#        (g_top["harm_score"] > 0) &
#        (g_top["clinical_plausibility"] >= 0.7)
#    ]
#
#    summary_rows.append({
#        "method": method,
#        "top_k_used": len(g_top),
#        "repairable_harmful_slices": len(repairable_harmful),
#        "worst_slice_AUPRC": g_top["AUPRC"].min(),
#        "worst_slice_recall": g_top["Recall"].min(),
#        "mean_harm_score": g_top["harm_score"].mean(),
#        "max_harm_score": g_top["harm_score"].max(),
#        "mean_clinical_plausibility": g_top["clinical_plausibility"].mean(),
#        "coverage_pct_naive": g_top["support_pct"].sum(),
#        "top_slice": g_top.iloc[0]["slice_name"]
#    })
#
#comparative_summary = pd.DataFrame(summary_rows)
#comparative_summary

precision of repairability

In [ ]:
TOP_K = 10
HARM_THRESHOLD = 0

summary_rows = []

for method, g in all_results.groupby("method"):
    g_ranked = g.sort_values("harm_score", ascending=False).head(TOP_K)

    summary_rows.append({
        "method": method,
        "top_k_used": len(g_ranked),
        "repairable_harmful_slices": int((g_ranked["harm_score"] > HARM_THRESHOLD).sum()),
        "worst_slice_AUPRC": g_ranked["AUPRC"].min(),
        "worst_slice_recall": g_ranked["Recall"].min(),
        "mean_harm_score": g_ranked["harm_score"].mean(),
        "max_harm_score": g_ranked["harm_score"].max(),
        "coverage_pct_naive": g_ranked["support_pct"].sum(),
        "top_slice": g_ranked.iloc[0]["slice_name"] if len(g_ranked) > 0 else None,
        "harm_per_slice": g_ranked["harm_score"].mean(),
    })

comparative_summary = pd.DataFrame(summary_rows)

comparative_summary["repairability_precision"] = (
    comparative_summary["repairable_harmful_slices"] /
    comparative_summary["top_k_used"].replace(0, np.nan)
)

comparative_summary = comparative_summary.sort_values(
    ["repairability_precision", "mean_harm_score"],
    ascending=False
)

comparative_summary

In [ ]:
#comparative_summary["repairability_precision"] = (
#    comparative_summary["repairable_harmful_slices"] /
#    comparative_summary["top_k_used"]
#)
#
#comparative_summary

method efficiency

In [ ]:
comparative_summary["harm_per_slice"] = (
    comparative_summary["mean_harm_score"] /
    comparative_summary["top_k_used"]
)

comparative_summary

Add stability

In [ ]:
#ef bootstrap_stability(data, slice_rule, n_boot=100, min_support=30):
#   harms = []
#
#   for b in range(n_boot):
#       boot = data.sample(frac=1.0, replace=True, random_state=42+b)
#       mask = slice_rule(boot)
#
#       m = evaluate_slice(boot, mask, "bootstrap", "tmp")
#       if m is not None:
#           harms.append(m["avg_logloss"])
#
#   if len(harms) < 10:
#       return np.nan
#
#   return 1 / (1 + np.std(harms))

In [ ]:
#ef get_slice_mask(data, slice_name):
#   """
#   Reconstruct slice membership for any slice name used by BASiC, SliceFinder-style,
#   SliceLine-style, or AFISP-inspired methods.
#   """
#   d = add_derived_slice_columns(data)
#   name = str(slice_name)
#
#   # AFISP broad phenotype rules
#   if name == "physiologic_instability":
#       return (
#           (d["tachycardia_flag"] == 1) |
#           (d["tachypnea_flag"] == 1) |
#           (d["hypoxia_flag"] == 1) |
#           (d["hypotension_flag"] == 1)
#       )
#
#   if name == "missing_vitals":
#       return d["vitals_missing_count"] >= 3
#
#   if name == "cardiorespiratory_risk":
#       return (
#           (d["chest_pain"] == 1) |
#           (d["pneumonia"] == 1) |
#           (d["chd"] == 1)
#       )
#
#   if name == "metabolic_comorbidity":
#       return (
#           (d["dm"] == 1) |
#           (d["ckd"] == 1)
#       )
#
#   if name == "complaint_missingness":
#       return (
#           ((d["abdominal_pain"] == 1) | (d["chest_pain"] == 1)) &
#           (d["vitals_missing_count"] >= 1)
#       )
#
#   # Convert SliceFinder/SliceLine style naming to BASiC naming
#   name = name.replace("_AND_", "_and_")
#
#   # BASiC accepted rules
#   temp_registry = pd.DataFrame({"slice_name": [name]})
#   temp_membership = build_slice_membership(d, temp_registry)
#
#   if name in temp_membership.columns:
#       return temp_membership[name] == 1
#
#   # Fallback: direct binary column
#   if name in d.columns:
#       return d[name] == 1
#
#   return pd.Series(False, index=d.index)
#
#
#ef bootstrap_stability(data, slice_name, n_boot=50):
#   """
#   Stability = how consistently harmful the slice remains under bootstrap resampling.
#   Higher is better.
#   """
#   harms = []
#
#   for b in range(n_boot):
#       boot = data.sample(frac=1.0, replace=True, random_state=42 + b)
#       mask = get_slice_mask(boot, slice_name)
#
#       m = evaluate_slice(boot, mask, "bootstrap", str(slice_name))
#
#       if m is not None:
#           m_df = add_harm_columns(pd.DataFrame([m]), global_ref)
#           harms.append(float(m_df["harm_score"].iloc[0]))
#
#   if len(harms) < 10:
#       return np.nan
#
#   harm_mean = np.mean(harms)
#   harm_std = np.std(harms)
#
#   return harm_mean / (harm_mean + harm_std + 1e-9)

In [ ]:
def get_slice_mask(data, slice_name):
    d = add_derived_slice_columns(data)
    name = str(slice_name).replace("_AND_", "_and_")

    # AFISP rules
    if name == "physiologic_instability":
        return (
            (d["tachycardia_flag"] == 1) |
            (d["tachypnea_flag"] == 1) |
            (d["hypoxia_flag"] == 1) |
            (d["hypotension_flag"] == 1)
        )

    if name == "missing_vitals":
        return d["vitals_missing_count"] >= 3

    if name == "cardiorespiratory_risk":
        return (
            (d["chest_pain"] == 1) |
            (d["pneumonia"] == 1) |
            (d["chd"] == 1)
        )

    if name == "metabolic_comorbidity":
        return (d["dm"] == 1) | (d["ckd"] == 1)

    if name == "complaint_missingness":
        return (
            ((d["abdominal_pain"] == 1) | (d["chest_pain"] == 1)) &
            (d["vitals_missing_count"] >= 1)
        )

    # Special BASiC aliases
    alias_rules = {
        "chest_pain_and_ambulance": lambda x: (x["chest_pain"] == 1) & (x["arrival_mode_ambulance"] == 1),
        "unclassified_and_vitals_missing_ge_3": lambda x: (x["unclassified"] == 1) & (x["vitals_missing_count"] >= 3),
        "vitals_missing_ge_3": lambda x: x["vitals_missing_count"] >= 3,
        "ambulance": lambda x: x["arrival_mode_ambulance"] == 1,
        "fever_temp": lambda x: x["fever_temp_flag"] == 1,
        "tachycardia": lambda x: x["tachycardia_flag"] == 1,
        "tachypnea": lambda x: x["tachypnea_flag"] == 1,
        "hypoxia": lambda x: x["hypoxia_flag"] == 1,
        "hypotension": lambda x: x["hypotension_flag"] == 1,
    }

    if name in alias_rules:
        return alias_rules[name](d)

    # Direct column
    if name in d.columns:
        return d[name] == 1

    # Pairwise or multi-condition rule
    if "_and_" in name:
        parts = name.split("_and_")
        mask = pd.Series(True, index=d.index)

        for p in parts:
            if p in alias_rules:
                mask &= alias_rules[p](d)
            elif p in d.columns:
                mask &= (d[p] == 1)
            else:
                return pd.Series(False, index=d.index)

        return mask

    return pd.Series(False, index=d.index)

In [ ]:
import numpy as np

def bootstrap_stability(df, slice_name, n_boot=50, random_state=42):
    """
    Bootstrap support stability for any method.
    Higher = slice membership remains consistent under resampling.
    """

    rng = np.random.default_rng(random_state)

    base_mask = get_slice_mask(df, slice_name).astype(bool)
    base_support = base_mask.mean()

    if base_support == 0:
        return np.nan

    supports = []
    n = len(df)

    for _ in range(n_boot):
        idx = rng.choice(n, n, replace=True)
        boot = df.iloc[idx].copy()
        boot_mask = get_slice_mask(boot, slice_name).astype(bool)
        supports.append(boot_mask.mean())

    supports = np.asarray(supports)

    cv = supports.std() / max(supports.mean(), 1e-8)
    stability = 1 - cv

    return float(np.clip(stability, 0, 1))

In [ ]:
stability_rows = []

for _, row in all_results.iterrows():
    s = row["slice_name"]
    method = row["method"]

    stability_rows.append({
        "method": method,
        "slice_name": s,
        "stability_score": bootstrap_stability(diag_train, s, n_boot=50)
    })

stability_df = pd.DataFrame(stability_rows)

all_results = all_results.merge(
    stability_df,
    on=["method", "slice_name"],
    how="left"
)

all_results[["method", "slice_name", "harm_score", "stability_score"]].sort_values(
    ["method", "stability_score"],
    ascending=[True, False]
).head(20)

In [ ]:
display(
    all_results.groupby("method").size()
)

In [ ]:
stability_check = (
    all_results
    .groupby("method")["stability_score"]
    .agg(["count", "mean", "min", "max"])
)

stability_check

In [ ]:
print("bootstrap_stability" in globals())

In [ ]:
def artifact_risk_score(slice_name):
    name = str(slice_name).lower()

    artifact_terms = [
        "unknown",
        "missing",
        "unclassified",
        "arrival_mode",
        "vitals_missing"
    ]

    clinical_terms = [
        "tachycardia",
        "tachypnea",
        "hypoxia",
        "hypotension",
        "pneumonia",
        "ckd",
        "dm",
        "chd",
        "chest",
        "abdominal"
    ]

    has_artifact = any(t in name for t in artifact_terms)
    has_clinical = any(t in name for t in clinical_terms)

    # pure operational/missingness slice = higher artifact risk
    if has_artifact and not has_clinical:
        return 1.0

    # mixed clinical + missingness slice = moderate artifact risk
    if has_artifact and has_clinical:
        return 0.5

    return 0.0


all_results["artifact_risk"] = all_results["slice_name"].apply(artifact_risk_score)
all_results["artifact_free"] = (all_results["artifact_risk"] <= 0.5).astype(int)

all_results[["method", "slice_name", "artifact_risk", "artifact_free"]].head(20)

In [ ]:
def jaccard_similarity(mask_a, mask_b):
    a = mask_a.astype(bool)
    b = mask_b.astype(bool)

    intersection = (a & b).sum()
    union = (a | b).sum()

    if union == 0:
        return 0.0

    return intersection / union


redundancy_rows = []

for method, g in all_results.groupby("method"):
    g_top = g.sort_values("harm_score", ascending=False).head(TOP_K).copy()
    names = g_top["slice_name"].tolist()

    max_redundancies = []

    for i, s1 in enumerate(names):
        mask1 = get_slice_mask(diag_train, s1)
        sims = []

        for j, s2 in enumerate(names):
            if i == j:
                continue

            mask2 = get_slice_mask(diag_train, s2)
            sims.append(jaccard_similarity(mask1, mask2))

        max_redundancies.append(max(sims) if sims else 0.0)

    temp = pd.DataFrame({
        "method": method,
        "slice_name": names,
        "redundancy_risk": max_redundancies
    })

    redundancy_rows.append(temp)

redundancy_df = pd.concat(redundancy_rows, ignore_index=True)

all_results = all_results.merge(
    redundancy_df,
    on=["method", "slice_name"],
    how="left"
)

all_results["redundancy_risk"] = all_results["redundancy_risk"].fillna(0.0)
all_results["unique_slice"] = (all_results["redundancy_risk"] < 0.5).astype(int)

all_results[["method", "slice_name", "redundancy_risk", "unique_slice"]].sort_values(
    ["method", "redundancy_risk"],
    ascending=[True, False]
).head(20)

In [ ]:
def true_coverage_pct(data, slice_names):
    covered = pd.Series(False, index=data.index)

    for s in slice_names:
        covered |= get_slice_mask(data, s).astype(bool)

    return 100 * covered.mean()

In [ ]:
#TOP_K = 10
#
#summary_rows = []
#
#for method, g in all_results.groupby("method"):
#    g_top = g.sort_values("harm_score", ascending=False).head(TOP_K)
#
#    repairable_harmful = g_top[
#        (g_top["harm_score"] > 0) &
#        (g_top["clinical_plausibility"] >= 0.7)
#    ]
#
#    artifact_free_harmful = g_top[
#        (g_top["harm_score"] > 0) &
#        (g_top["artifact_free"] == 1)
#    ]
#
#    unique_harmful = g_top[
#        (g_top["harm_score"] > 0) &
#        (g_top["unique_slice"] == 1)
#    ]
#
#    summary_rows.append({
#        "method": method,
#        "top_k_used": len(g_top),
#        "repairable_harmful_slices": len(repairable_harmful),
#        "worst_slice_AUPRC": g_top["AUPRC"].min(),
#        "worst_slice_recall": g_top["Recall"].min(),
#        "mean_harm_score": g_top["harm_score"].mean(),
#        "max_harm_score": g_top["harm_score"].max(),
#        "mean_clinical_plausibility": g_top["clinical_plausibility"].mean(),
#        "mean_stability_score": g_top["stability_score"].mean(),
#        "artifact_free_precision": len(artifact_free_harmful) / len(g_top),
#        "unique_slice_fraction": len(unique_harmful) / len(g_top),
#        "coverage_pct_true": true_coverage_pct(diag_train, g_top["slice_name"].tolist()),
#        "coverage_pct_naive": g_top["support_pct"].sum(),
#        "mean_redundancy_risk": g_top["redundancy_risk"].mean(),
#        "top_slice": g_top.iloc[0]["slice_name"]
#    })
#
#comparative_summary = pd.DataFrame(summary_rows)
#
#comparative_summary["repairability_precision"] = (
#    comparative_summary["repairable_harmful_slices"] /
#    comparative_summary["top_k_used"]
#)
#
#comparative_summary["harm_per_slice"] = (
#    comparative_summary["mean_harm_score"] /
#    comparative_summary["top_k_used"]
#)
#
#comparative_summary

In [ ]:
# ============================================================
# COMPLETE ASAP-CONSISTENT COMPARATIVE SUMMARY
#
# Applied equally to:
# - BASiC
# - SliceFinder-style
# - SliceLine-style
# - AFISP-inspired
#
# Because Notebook 05b has no separate temporal-persistence
# measurement, bootstrap stability is used as the combined
# replication/stability criterion.
# ============================================================

import numpy as np
import pandas as pd


TOP_K = 10

STABILITY_THRESHOLD = 0.60
PLAUSIBILITY_THRESHOLD = 0.70
REDUNDANCY_THRESHOLD = 0.50


comparison_pool = (
    all_results.copy()
)


# ------------------------------------------------------------
# 1. Support adequacy
# ------------------------------------------------------------

comparison_pool[
    "asap_support_pass"
] = (
    comparison_pool["n"]
    .ge(MIN_SUPPORT_N)
    .astype(int)
)


# ------------------------------------------------------------
# 2. Positive composite harm
# ------------------------------------------------------------

comparison_pool[
    "asap_harm_pass"
] = (
    comparison_pool[
        "harm_score"
    ]
    .gt(0)
    .astype(int)
)


# ------------------------------------------------------------
# 3. Combined replication/stability criterion
# ------------------------------------------------------------

comparison_pool[
    "asap_replication_stability_pass"
] = (
    pd.to_numeric(
        comparison_pool[
            "stability_score"
        ],
        errors="coerce",
    )
    .ge(STABILITY_THRESHOLD)
    .fillna(False)
    .astype(int)
)


# ------------------------------------------------------------
# 4. Clinical plausibility
# ------------------------------------------------------------

comparison_pool[
    "asap_plausibility_pass"
] = (
    pd.to_numeric(
        comparison_pool[
            "clinical_plausibility"
        ],
        errors="coerce",
    )
    .ge(PLAUSIBILITY_THRESHOLD)
    .fillna(False)
    .astype(int)
)


# ------------------------------------------------------------
# 5. Artifact control
# ------------------------------------------------------------

comparison_pool[
    "asap_artifact_pass"
] = (
    comparison_pool[
        "artifact_free"
    ]
    .fillna(0)
    .astype(int)
)


# ------------------------------------------------------------
# 6. Redundancy control
# ------------------------------------------------------------

comparison_pool[
    "asap_redundancy_pass"
] = (
    pd.to_numeric(
        comparison_pool[
            "redundancy_risk"
        ],
        errors="coerce",
    )
    .lt(REDUNDANCY_THRESHOLD)
    .fillna(False)
    .astype(int)
)


# ------------------------------------------------------------
# 7. Reconstructable membership
# ------------------------------------------------------------

membership_available = []

for slice_name in (
    comparison_pool[
        "slice_name"
    ].astype(str)
):
    try:
        mask = get_slice_mask(
            diag_train,
            slice_name,
        )

        available = int(
            np.asarray(
                mask
            ).astype(bool).sum()
            > 0
        )

    except Exception:
        available = 0

    membership_available.append(
        available
    )


comparison_pool[
    "asap_membership_pass"
] = membership_available


# ------------------------------------------------------------
# 8. Complete comparison eligibility
# ------------------------------------------------------------

complete_asap_columns = [
    "asap_support_pass",
    "asap_harm_pass",
    "asap_replication_stability_pass",
    "asap_plausibility_pass",
    "asap_artifact_pass",
    "asap_redundancy_pass",
    "asap_membership_pass",
]


comparison_pool[
    "complete_asap_eligible"
] = (
    comparison_pool[
        complete_asap_columns
    ]
    .fillna(0)
    .astype(int)
    .all(axis=1)
    .astype(int)
)


# ------------------------------------------------------------
# 9. Standardized TOP10 comparison
# ------------------------------------------------------------

summary_rows = []


for method, method_group in (
    comparison_pool.groupby(
        "method"
    )
):
    method_top = (
        method_group
        .sort_values(
            "harm_score",
            ascending=False,
        )
        .head(TOP_K)
        .copy()
    )

    eligible_top = (
        method_top.loc[
            method_top[
                "complete_asap_eligible"
            ].eq(1)
        ]
    )

    top_k_used = int(
        len(method_top)
    )

    n_complete_asap = int(
        len(eligible_top)
    )

    repairability_precision = (
        n_complete_asap
        / top_k_used
        if top_k_used > 0
        else np.nan
    )

    support_quality = float(
        method_top[
            "asap_support_pass"
        ].mean()
    )

    replication_stability_quality = float(
        method_top[
            "asap_replication_stability_pass"
        ].mean()
    )

    plausibility_quality = float(
        method_top[
            "asap_plausibility_pass"
        ].mean()
    )

    artifact_quality = float(
        method_top[
            "asap_artifact_pass"
        ].mean()
    )

    redundancy_quality = float(
        method_top[
            "asap_redundancy_pass"
        ].mean()
    )

    membership_quality = float(
        method_top[
            "asap_membership_pass"
        ].mean()
    )

    governance_score = float(
        np.mean([
            repairability_precision,
            support_quality,
            replication_stability_quality,
            plausibility_quality,
            artifact_quality,
            redundancy_quality,
            membership_quality,
        ])
    )

    summary_rows.append({
        "method":
            method,

        "top_k_used":
            top_k_used,

        "complete_ASAP_eligible_slices":
            n_complete_asap,

        "repairable_harmful_slices":
            n_complete_asap,

        "repairability_precision":
            repairability_precision,

        "worst_slice_AUPRC":
            method_top[
                "AUPRC"
            ].min(),

        "worst_slice_recall":
            method_top[
                "Recall"
            ].min(),

        "mean_harm_score":
            method_top[
                "harm_score"
            ].mean(),

        "max_harm_score":
            method_top[
                "harm_score"
            ].max(),

        "support_adequacy":
            support_quality,

        "replication_stability":
            replication_stability_quality,

        "clinical_plausibility":
            plausibility_quality,

        "artifact_control":
            artifact_quality,

        "redundancy_control":
            redundancy_quality,

        "membership_availability":
            membership_quality,

        "coverage_pct_true":
            true_coverage_pct(
                diag_train,
                method_top[
                    "slice_name"
                ].tolist(),
            ),

        "governance_score":
            governance_score,

        "top_slice":
            (
                method_top
                .iloc[0][
                    "slice_name"
                ]
                if len(method_top)
                else None
            ),
    })


comparative_summary_complete_asap = (
    pd.DataFrame(
        summary_rows
    )
    .sort_values(
        [
            "governance_score",
            "repairability_precision",
            "mean_harm_score",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)


display(
    comparative_summary_complete_asap
)


comparison_pool.to_csv(
    "05b_complete_ASAP_candidate_audit.csv",
    index=False,
)

comparative_summary_complete_asap.to_csv(
    "05b_comparative_summary_complete_ASAP.csv",
    index=False,
)


print(
    "Complete ASAP comparative summary saved."
)

In [ ]:
sensitivity_rows = []

for K in [5, 10, 15]:
    for method, g in all_results.groupby("method"):
        g_top = g.sort_values("harm_score", ascending=False).head(K)

        repairable = g_top[
            (g_top["harm_score"] > 0) &
            (g_top["clinical_plausibility"] >= 0.7)
        ]

        sensitivity_rows.append({
            "K": K,
            "method": method,
            "repairability_precision": len(repairable) / len(g_top),
            "mean_harm_score": g_top["harm_score"].mean(),
            "mean_clinical_plausibility": g_top["clinical_plausibility"].mean(),
            "mean_stability_score": g_top["stability_score"].mean(),
            "artifact_free_precision": (g_top["artifact_free"] == 1).mean(),
            "unique_slice_fraction": (g_top["unique_slice"] == 1).mean(),
            "coverage_pct_true": true_coverage_pct(diag_train, g_top["slice_name"].tolist())
        })

topk_sensitivity = pd.DataFrame(sensitivity_rows)
topk_sensitivity

In [ ]:
#comparative_summary["governance_score"] = (
#    0.30 * comparative_summary["repairability_precision"]
#    + 0.25 * comparative_summary["mean_clinical_plausibility"]
#    + 0.20 * comparative_summary["mean_stability_score"].fillna(0)
#    + 0.15 * comparative_summary["artifact_free_precision"]
#    + 0.10 * (1 - comparative_summary["mean_redundancy_risk"])
#)
#
#comparative_summary.sort_values("governance_score", ascending=False)

Save final outputs for Section VI

In [ ]:
#all_results.to_csv("comparative_all_slice_discovery_results.csv", index=False)
#comparative_summary.to_csv("table_X_comparative_method_summary.csv", index=False)
#
#top_slices = (
#    all_results
#    .sort_values(["method", "harm_score"], ascending=[True, False])
#    .groupby("method")
#    .head(10)
#)
#
#top_slices.to_csv("table_IX_top_slices_by_method.csv", index=False)

In [ ]:
#all_results.to_csv(
#    OUT_DIR / "comparative_all_slice_discovery_results.csv",
#    index=False
#)
#
#comparative_summary.to_csv(
#    OUT_DIR / "table_X_comparative_method_summary.csv",
#    index=False
#)
#
#top_slices = (
#    all_results
#    .sort_values(["method", "harm_score"], ascending=[True, False])
#    .groupby("method")
#    .head(10)
#)
#
#top_slices.to_csv(
#    OUT_DIR / "table_IX_top_slices_by_method.csv",
#    index=False
#)
#
#print("Saved outputs to:", OUT_DIR)

In [ ]:
#final_table_X = comparative_summary.sort_values(
#    "governance_score",
#    ascending=False
#).copy()
#
#final_table_X.to_csv(
#    OUT_DIR / "FINAL_Table_X_Governance_Ranked_Method_Comparison.csv",
#    index=False
#)
#
#final_table_X

In [ ]:
final_table_X = (
    comparative_summary_complete_asap
    .sort_values(
        "governance_score",
        ascending=False,
    )
    .copy()
)

final_table_X.to_csv(
    OUT_DIR
    / "FINAL_Table_X_Governance_Ranked_Method_Comparison.csv",
    index=False,
)

display(final_table_X)

In [ ]:
display(
    comparative_summary_complete_asap[
        [
            "method",
            "top_k_used",
            "complete_ASAP_eligible_slices",
            "repairability_precision",
            "governance_score",
        ]
    ]
)

In [ ]:
t_runtime_agreement = start_timer(
    "Cross-method agreement analysis"
)
def canonical_slice_name(name):
    name = str(name).lower()
    name = name.replace("_and_", "_AND_")
    name = name.replace("_AND_", "_and_")

    aliases = {
        "abdominal_pain_and_hr_missing": "abdominal_pain_and_hr_missing",
        "abdominal_pain_AND_hr_missing": "abdominal_pain_and_hr_missing",
        "ckd_AND_spo2_missing": "ckd_and_spo2_missing",
        "ckd_and_spo2_missing": "ckd_and_spo2_missing",
        "tachycardia_flag": "tachycardia",
        "tachycardia": "tachycardia",
        "tachypnea_flag": "tachypnea",
        "tachypnea": "tachypnea",
        "hypoxia_flag": "hypoxia",
        "hypoxia": "hypoxia",
        "chest_pain_AND_arrival_mode_ambulance": "chest_pain_and_ambulance",
        "chest_pain_and_ambulance": "chest_pain_and_ambulance",
    }

    return aliases.get(name, name)

In [ ]:
agreement_base = all_results.copy()
agreement_base["canonical_slice"] = agreement_base["slice_name"].apply(canonical_slice_name)

# Use top-K slices per method, consistent with Table X
topk_for_agreement = (
    agreement_base
    .sort_values(["method", "harm_score"], ascending=[True, False])
    .groupby("method")
    .head(TOP_K)
)

agreement_table = (
    topk_for_agreement
    .assign(present=1)
    .pivot_table(
        index="canonical_slice",
        columns="method",
        values="present",
        aggfunc="max",
        fill_value=0
    )
    .reset_index()
)

agreement_table

In [ ]:
methods = sorted(topk_for_agreement["method"].unique())

basic_method = "BASiC_ASAP"

basic_slices = set(
    topk_for_agreement.loc[
        topk_for_agreement["method"] == basic_method,
        "canonical_slice"
    ]
)

competitor_methods = [m for m in methods if m != basic_method]

competitor_slices = {
    m: set(topk_for_agreement.loc[topk_for_agreement["method"] == m, "canonical_slice"])
    for m in competitor_methods
}

all_competitor_slices = set().union(*competitor_slices.values())

basic_rediscovered_by_any = basic_slices & all_competitor_slices
basic_unique = basic_slices - all_competitor_slices

agreement_summary_rows = []

agreement_summary_rows.append({
    "metric": "BASiC slices rediscovered by any competitor",
    "numerator": len(basic_rediscovered_by_any),
    "denominator": len(basic_slices),
    "percentage": 100 * len(basic_rediscovered_by_any) / max(len(basic_slices), 1)
})

agreement_summary_rows.append({
    "metric": "BASiC slices unique to BASiC",
    "numerator": len(basic_unique),
    "denominator": len(basic_slices),
    "percentage": 100 * len(basic_unique) / max(len(basic_slices), 1)
})

for m, sset in competitor_slices.items():
    accepted_by_basic = sset & basic_slices

    agreement_summary_rows.append({
        "metric": f"{m} slices accepted by BASiC",
        "numerator": len(accepted_by_basic),
        "denominator": len(sset),
        "percentage": 100 * len(accepted_by_basic) / max(len(sset), 1)
    })

agreement_summary = pd.DataFrame(agreement_summary_rows)
agreement_summary

In [ ]:
agreement_table.to_csv(
    OUT_DIR / "table_cross_method_slice_agreement.csv",
    index=False
)

agreement_summary.to_csv(
    OUT_DIR / "table_cross_method_agreement_summary.csv",
    index=False
)

print("Saved cross-method agreement outputs to:", OUT_DIR)

In [ ]:
agreement_table.to_csv(
    OUT_DIR / "FINAL_Table_XII_Cross_Method_Slice_Agreement.csv",
    index=False
)

agreement_summary.to_csv(
    OUT_DIR / "FINAL_Table_XIII_Cross_Method_Agreement_Summary.csv",
    index=False
)
stop_timer(
    "Cross-method agreement analysis",
    t_runtime_agreement
)

In [ ]:
all_results.to_csv(
    OUT_DIR / "comparative_all_slice_discovery_results_with_stability_artifact_redundancy.csv",
    index=False
)

comparative_summary.to_csv(
    OUT_DIR / "table_X_comparative_method_summary_with_governance.csv",
    index=False
)

top_slices = (
    all_results
    .sort_values(["method", "harm_score"], ascending=[True, False])
    .groupby("method")
    .head(10)
)

top_slices.to_csv(
    OUT_DIR / "table_IX_top_slices_by_method_with_governance.csv",
    index=False
)

print("Saved governance-aware outputs to:", OUT_DIR)

In [ ]:
topk_sensitivity.to_csv(
    OUT_DIR / "table_topk_sensitivity_comparative_methods.csv",
    index=False
)

In [ ]:
comparative_summary.to_csv(
    "comparative_summary.csv",
    index=False
)

##########################################################
SECTION 10
Governance Evaluation
##########################################################

In [ ]:
# ================================================================
# SECTION 10 — GOVERNANCE EVALUATION
# Table XI + Figure 8
# Single-cell, reproducible implementation
# ================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

# ---------------------------------------------------------------
# 1. Check that the required upstream tables exist
# ---------------------------------------------------------------

required_objects = ["comparative_summary", "all_results"]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "The following required objects are not available: "
        f"{missing_objects}\n\n"
        "Run the preceding Notebook 05b comparative-analysis cells first. "
        "This governance section requires both `comparative_summary` and "
        "`all_results`."
    )

comparison = (
    comparative_summary_complete_asap.copy()
)
slice_results = all_results.copy()

# ---------------------------------------------------------------
# 2. Standardize column names
# ---------------------------------------------------------------

def standardize_columns(df):
    """Return a copy with standardized lowercase column names."""

    df = df.copy()

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[\s\-/]+", "_", regex=True)
        .str.replace(r"[^a-z0-9_]", "", regex=True)
        .str.replace(r"_+", "_", regex=True)
        .str.strip("_")
    )

    return df


comparison = standardize_columns(comparison)
slice_results = standardize_columns(slice_results)

print("=" * 76)
print("SECTION 10 — GOVERNANCE EVALUATION")
print("=" * 76)

print("\nColumns detected in comparative_summary:")
print(comparison.columns.tolist())

print("\nColumns detected in all_results:")
print(slice_results.columns.tolist())

# ---------------------------------------------------------------
# 3. Robust column finder
# ---------------------------------------------------------------

def find_column(df, aliases, description, required=True):
    """
    Find a column using exact aliases first and conservative partial
    matching second.
    """

    columns = list(df.columns)

    # Exact matching
    for alias in aliases:
        if alias in columns:
            return alias

    # Conservative partial matching
    for alias in aliases:
        matches = [
            col for col in columns
            if alias in col or col in alias
        ]

        if len(matches) == 1:
            return matches[0]

    if required:
        raise ValueError(
            f"\nCould not identify the column for '{description}'.\n"
            f"Accepted alternatives: {aliases}\n"
            f"Available columns: {columns}"
        )

    return None


# Comparative-summary columns
comparison_method_col = find_column(
    comparison,
    [
        "method",
        "method_name",
        "discovery_method",
        "slice_discovery_method"
    ],
    "method in comparative_summary"
)

plausibility_col = find_column(
    comparison,
    [
        "mean_clinical_plausibility",
        "mean_plausibility",
        "clinical_plausibility",
        "avg_clinical_plausibility",
        "average_clinical_plausibility",
        "plausibility_score",
        "mean_plausibility_score",
        "plausibility"
    ],
    "clinical plausibility"
)

repairability_col = find_column(
    comparison,
    [
        "repairability_precision",
        "repairable_precision",
        "repair_precision",
        "precision_repairability"
    ],
    "repairability precision"
)

# Slice-level all_results columns
slice_method_col = find_column(
    slice_results,
    [
        "method",
        "method_name",
        "discovery_method",
        "slice_discovery_method"
    ],
    "method in all_results"
)

stability_col = find_column(
    slice_results,
    [
        "bootstrap_stability",
        "bootstrap_stability_score",
        "stability_score",
        "mean_stability",
        "stability"
    ],
    "slice stability"
)

artifact_col = find_column(
    slice_results,
    [
        "artifact_flag",
        "artifact_risk",
        "artifact_indicator",
        "is_artifact",
        "artifact"
    ],
    "artifact flag or artifact risk"
)

redundancy_col = find_column(
    slice_results,
    [
        "redundancy_flag",
        "redundancy_risk",
        "redundancy_indicator",
        "is_redundant",
        "redundancy"
    ],
    "redundancy flag or redundancy risk"
)

print("\nColumns selected:")
print(f"  Comparative method:      {comparison_method_col}")
print(f"  Clinical plausibility:    {plausibility_col}")
print(f"  Repairability precision: {repairability_col}")
print(f"  Slice-level method:       {slice_method_col}")
print(f"  Stability:                {stability_col}")
print(f"  Artifact indicator:       {artifact_col}")
print(f"  Redundancy indicator:     {redundancy_col}")

# ---------------------------------------------------------------
# 4. Canonicalize method names
# ---------------------------------------------------------------

def canonical_method_name(value):
    """Map alternative method labels to the four manuscript labels."""

    text = (
        str(value)
        .strip()
        .lower()
        .replace("-", "_")
        .replace("/", "_")
        .replace(" ", "_")
    )

    if "basic" in text or "asap" in text:
        return "BASiC_ASAP"

    if "afisp" in text:
        return "AFISP_inspired"

    if "slicefinder" in text or "slice_finder" in text:
        return "SliceFinder_style"

    if "sliceline" in text or "slice_line" in text:
        return "SliceLine_style"

    return str(value)


comparison["method_standard"] = comparison[
    comparison_method_col
].apply(canonical_method_name)

slice_results["method_standard"] = slice_results[
    slice_method_col
].apply(canonical_method_name)

required_methods = [
    "AFISP_inspired",
    "SliceFinder_style",
    "SliceLine_style",
    "BASiC_ASAP"
]

display_labels = {
    "AFISP_inspired": "AFISP-inspired",
    "SliceFinder_style": "SliceFinder-style",
    "SliceLine_style": "SliceLine-style",
    "BASiC_ASAP": "BASiC / ASAP"
}

# ---------------------------------------------------------------
# 5. Convert governance fields to numeric scores
# ---------------------------------------------------------------

def numeric_score(series, field_name):
    """
    Convert a numeric score to [0,1].

    Values stored as percentages, such as 95, are converted to 0.95.
    """

    result = pd.to_numeric(series, errors="coerce")

    if result.notna().sum() == 0:
        raise ValueError(
            f"'{field_name}' contains no usable numeric values."
        )

    finite_max = result.dropna().max()

    if finite_max > 1 and finite_max <= 100:
        result = result / 100.0

    return result.clip(0, 1)


def risk_indicator(series, field_name):
    """
    Convert an artifact/redundancy field to a risk value in [0,1].

    Interpretation:
        0.00 = no identified risk
        0.50 = possible/moderate/uncertain risk
        1.00 = identified or likely risk
    """

    # Preserve genuinely numeric values
    numeric = pd.to_numeric(series, errors="coerce")

    output = pd.Series(
        np.nan,
        index=series.index,
        dtype=float
    )

    numeric_mask = numeric.notna()

    if numeric_mask.any():
        numeric_values = numeric.loc[numeric_mask].astype(float)

        if numeric_values.max() > 1 and numeric_values.max() <= 100:
            numeric_values = numeric_values / 100.0

        output.loc[numeric_mask] = numeric_values.clip(0, 1)

    text = (
        series.astype(str)
        .str.strip()
        .str.lower()
    )

    false_values = {
        "false", "no", "n", "0",
        "none", "absent", "unlikely",
        "low", "low_risk", "not_flagged",
        "acceptable", "passed", "pass"
    }

    intermediate_values = {
        "possible", "moderate", "medium",
        "uncertain", "review", "partial"
    }

    true_values = {
        "true", "yes", "y", "1",
        "present", "likely", "high",
        "high_risk", "flagged", "fail", "failed"
    }

    output.loc[text.isin(false_values)] = 0.0
    output.loc[text.isin(intermediate_values)] = 0.5
    output.loc[text.isin(true_values)] = 1.0

    unresolved = output.isna() & series.notna()

    if unresolved.any():
        unresolved_values = sorted(
            text.loc[unresolved].unique().tolist()
        )

        raise ValueError(
            f"\nUnrecognized values in '{field_name}': "
            f"{unresolved_values}\n\n"
            "Update the mappings inside risk_indicator() before "
            "calculating governance scores."
        )

    return output


comparison["clinical_plausibility"] = numeric_score(
    comparison[plausibility_col],
    "clinical plausibility"
)

comparison["repairability_precision"] = numeric_score(
    comparison[repairability_col],
    "repairability precision"
)

slice_results["stability"] = numeric_score(
    slice_results[stability_col],
    "stability"
)

slice_results["artifact_risk_numeric"] = risk_indicator(
    slice_results[artifact_col],
    "artifact indicator"
)

slice_results["redundancy_risk_numeric"] = risk_indicator(
    slice_results[redundancy_col],
    "redundancy indicator"
)

# ---------------------------------------------------------------
# 6. Aggregate comparative-summary dimensions by method
# ---------------------------------------------------------------

comparative_governance = (
    comparison[
        comparison["method_standard"].isin(required_methods)
    ]
    .groupby("method_standard", as_index=False)
    .agg(
        clinical_plausibility=(
            "clinical_plausibility",
            "mean"
        ),
        repairability_precision=(
            "repairability_precision",
            "mean"
        )
    )
)

# ---------------------------------------------------------------
# 7. Aggregate slice-level governance dimensions by method
# ---------------------------------------------------------------

slice_governance = (
    slice_results[
        slice_results["method_standard"].isin(required_methods)
    ]
    .groupby("method_standard", as_index=False)
    .agg(
        stability=("stability", "mean"),
        artifact_risk=(
            "artifact_risk_numeric",
            "mean"
        ),
        redundancy_risk=(
            "redundancy_risk_numeric",
            "mean"
        ),
        evaluated_slices=(
            "method_standard",
            "size"
        )
    )
)

# Control is the complement of the observed risk rate
slice_governance["artifact_control"] = (
    1.0 - slice_governance["artifact_risk"]
)

slice_governance["redundancy_control"] = (
    1.0 - slice_governance["redundancy_risk"]
)

# ---------------------------------------------------------------
# 8. Merge the governance dimensions
# ---------------------------------------------------------------

governance_summary = comparative_governance.merge(
    slice_governance[
        [
            "method_standard",
            "stability",
            "artifact_control",
            "redundancy_control",
            "evaluated_slices"
        ]
    ],
    on="method_standard",
    how="outer",
    validate="one_to_one"
)

# Confirm all required methods are represented
missing_methods = [
    method for method in required_methods
    if method not in governance_summary[
        "method_standard"
    ].values
]

if missing_methods:
    raise ValueError(
        "\nGovernance information is incomplete for these methods:\n"
        f"{missing_methods}\n\n"
        "Check that both comparative_summary and all_results contain "
        "rows for AFISP-inspired, SliceFinder-style, SliceLine-style, "
        "and BASiC/ASAP."
    )

governance_dimensions = [
    "clinical_plausibility",
    "repairability_precision",
    "stability",
    "artifact_control",
    "redundancy_control"
]

# Stop rather than silently fabricating missing governance values
if governance_summary[governance_dimensions].isna().any().any():

    problem_rows = governance_summary.loc[
        governance_summary[
            governance_dimensions
        ].isna().any(axis=1),
        ["method_standard"] + governance_dimensions
    ]

    print("\nIncomplete governance rows:")
    display(problem_rows)

    raise ValueError(
        "One or more governance dimensions are missing. "
        "The calculation has stopped to prevent unsupported values "
        "from being inserted into Table XI or Figure 8."
    )

# Keep all scores within the intended normalized range
for column in governance_dimensions:
    governance_summary[column] = (
        governance_summary[column]
        .astype(float)
        .clip(0, 1)
    )

# ---------------------------------------------------------------
# 9. Derive the overall Governance Score
# ---------------------------------------------------------------
# Equal weighting is used because the governance dimensions represent
# distinct acceptance requirements and no empirically validated
# alternative governance weighting has yet been established.

governance_summary["governance_score"] = (
    governance_summary[
        governance_dimensions
    ].mean(axis=1)
)

# Arrange methods in manuscript order
governance_summary["method_standard"] = pd.Categorical(
    governance_summary["method_standard"],
    categories=required_methods,
    ordered=True
)

governance_summary = (
    governance_summary
    .sort_values("method_standard")
    .reset_index(drop=True)
)

governance_summary["method"] = governance_summary[
    "method_standard"
].astype(str).map(display_labels)

# ---------------------------------------------------------------
# 10. Prepare Table XI
# ---------------------------------------------------------------

table_columns = [
    "method",
    "governance_score",
    "clinical_plausibility",
    "repairability_precision",
    "stability",
    "artifact_control",
    "redundancy_control",
    "evaluated_slices"
]

table_xi = governance_summary[table_columns].copy()

table_xi.columns = [
    "Method",
    "Governance Score",
    "Clinical Plausibility",
    "Repairability Precision",
    "Stability",
    "Artifact Control",
    "Redundancy Control",
    "Evaluated Slices"
]

score_columns = [
    "Governance Score",
    "Clinical Plausibility",
    "Repairability Precision",
    "Stability",
    "Artifact Control",
    "Redundancy Control"
]

table_xi_display = table_xi.copy()

for column in score_columns:
    table_xi_display[column] = (
        table_xi_display[column]
        .astype(float)
        .round(3)
    )

table_xi_display["Evaluated Slices"] = (
    table_xi_display["Evaluated Slices"]
    .astype(int)
)

print("\n")
print("=" * 76)
print("TABLE XI — GOVERNANCE EVALUATION OF SLICE DISCOVERY METHODS")
print("=" * 76)

display(
    table_xi_display.style
    .format(
        {
            column: "{:.3f}"
            for column in score_columns
        }
    )
    .set_properties(
        **{
            "text-align": "center"
        }
    )
    .set_properties(
        subset=["Method"],
        **{
            "text-align": "left",
            "font-weight": "bold"
        }
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("text-align", "center"),
                    ("font-weight", "bold")
                ]
            }
        ]
    )
)

# ---------------------------------------------------------------
# 11. Export Table XI and the Figure 8 source data
# ---------------------------------------------------------------

table_xi.to_csv(
    "Table_XI_Governance_Evaluation.csv",
    index=False
)

governance_summary[
    [
        "method",
        "governance_score",
        "clinical_plausibility",
        "repairability_precision",
        "stability",
        "artifact_control",
        "redundancy_control"
    ]
].to_csv(
    "governance_summary.csv",
    index=False
)

# Optional LaTeX export for the manuscript
with open(
    "Table_XI_Governance_Evaluation.tex",
    "w",
    encoding="utf-8"
) as latex_file:

    latex_file.write(
        table_xi.to_latex(
            index=False,
            float_format="%.3f",
            caption=(
                "Governance evaluation of competing slice "
                "discovery methods."
            ),
            label="tab:governance_evaluation"
        )
    )

# ---------------------------------------------------------------
# 12. Figure 8 radar-chart configuration
# ---------------------------------------------------------------

radar_dimensions = [
    "governance_score",
    "clinical_plausibility",
    "repairability_precision",
    "stability",
    "artifact_control",
    "redundancy_control"
]

radar_labels = [
    "Governance\nScore",
    "Clinical\nPlausibility",
    "Repairability\nPrecision",
    "Stability",
    "Artifact\nControl",
    "Redundancy\nControl"
]

number_of_axes = len(radar_labels)

angles = np.linspace(
    0,
    2 * np.pi,
    number_of_axes,
    endpoint=False
).tolist()

angles += angles[:1]

# Draw BASiC last to keep its balanced polygon visible
radar_order = [
    "AFISP-inspired",
    "SliceFinder-style",
    "SliceLine-style",
    "BASiC / ASAP"
]

radar_df = (
    governance_summary
    .set_index("method")
    .loc[radar_order]
    .reset_index()
)

# Okabe–Ito colour-blind-safe palette
colour_map = {
    "AFISP-inspired": "#0072B2",
    "SliceFinder-style": "#009E73",
    "SliceLine-style": "#CC79A7",
    "BASiC / ASAP": "#D55E00"
}

# Additional encodings improve grayscale readability
line_style_map = {
    "AFISP-inspired": "--",
    "SliceFinder-style": "-.",
    "SliceLine-style": ":",
    "BASiC / ASAP": "-"
}

marker_map = {
    "AFISP-inspired": "s",
    "SliceFinder-style": "^",
    "SliceLine-style": "D",
    "BASiC / ASAP": "o"
}

# ---------------------------------------------------------------
# 13. Generate Figure 8
# ---------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(9.4, 8.8),
    subplot_kw={"polar": True}
)

# Start at the top and proceed clockwise
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)

for _, row in radar_df.iterrows():

    method = row["method"]

    values = (
        row[radar_dimensions]
        .astype(float)
        .tolist()
    )

    values += values[:1]

    is_basic = method == "BASiC / ASAP"

    ax.plot(
        angles,
        values,
        color=colour_map[method],
        linewidth=3.6 if is_basic else 2.0,
        linestyle=line_style_map[method],
        marker=marker_map[method],
        markersize=5.8 if is_basic else 4.6,
        markeredgecolor="black",
        markeredgewidth=0.4,
        label=method,
        zorder=6 if is_basic else 3
    )

    ax.fill(
        angles,
        values,
        color=colour_map[method],
        alpha=0.11 if is_basic else 0.04,
        zorder=2 if is_basic else 1
    )

# ---------------------------------------------------------------
# 14. Radar formatting
# ---------------------------------------------------------------

ax.set_xticks(angles[:-1])

ax.set_xticklabels(
    radar_labels,
    fontsize=10
)

ax.tick_params(
    axis="x",
    pad=15
)

ax.set_ylim(0, 1)

ax.set_yticks(
    [0.2, 0.4, 0.6, 0.8, 1.0]
)

ax.set_yticklabels(
    ["0.2", "0.4", "0.6", "0.8", "1.0"],
    fontsize=8
)

ax.set_rlabel_position(82)

ax.grid(
    linewidth=0.8,
    alpha=0.35
)

ax.spines["polar"].set_linewidth(0.8)

ax.set_title(
    "Governance Comparison of Slice Discovery Methods",
    fontsize=13,
    fontweight="bold",
    pad=48
)

ax.text(
    0.5,
    1.105,
    "Higher normalized values indicate stronger governance performance",
    transform=ax.transAxes,
    ha="center",
    va="center",
    fontsize=9,
    color="dimgray"
)

ax.legend(
    loc="upper left",
    bbox_to_anchor=(1.04, 1.04),
    frameon=False,
    fontsize=9.5,
    handlelength=3
)

# Reserve space for the external legend
fig.subplots_adjust(
    left=0.08,
    right=0.78,
    top=0.86,
    bottom=0.08
)

# ---------------------------------------------------------------
# 15. Save before displaying
# ---------------------------------------------------------------

fig.savefig(
    "Fig8_Governance_Comparison_Radar.pdf",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    "Fig8_Governance_Comparison_Radar.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# ---------------------------------------------------------------
# 16. Completion report
# ---------------------------------------------------------------

print("\n")
print("=" * 76)
print("GOVERNANCE EVALUATION COMPLETED")
print("=" * 76)

print("\nGenerated files:")
print("  Table_XI_Governance_Evaluation.csv")
print("  Table_XI_Governance_Evaluation.tex")
print("  governance_summary.csv")
print("  Fig8_Governance_Comparison_Radar.pdf")
print("  Fig8_Governance_Comparison_Radar.png")

In [ ]:
# =====================================================
# Export cross-method slice registry
# =====================================================

cross_method_registry = (
    all_results[
        [
            "method",
            "slice_name"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["method", "slice_name"]
    )
)

cross_method_registry.to_csv(
    "cross_method_registry.csv",
    index=False
)

print(
    f"Exported {len(cross_method_registry)} "
    "method-slice pairs."
)